In [1]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath('..')) # Go up one level to the project root and add it to the path
from src.etl.ingest import load_brfss
# If you just want to see the whole data config dictionary
#from src.utils.config import load_config
# data_config = load_config("data.yaml")

In [ ]:
#TODO Figure out what the variable means and create the fuction (maybe not in ingest but in a separate file)

In [4]:
# load the RAW BRFSS data from ONE DRIVE, it reads the path from the .env file, but you can also specify it directly as an argument to load_brfss()
brfss_df = load_brfss(2024)

In [6]:
cols = ['_STATE', 'ADDEPEV3', '_LLCPWT'] 
df = brfss_df[cols].copy()

# --- Print prevalence of excluded values ---
total = len(df)
missing = df['ADDEPEV3'].isna().sum()
dont_know = (df['ADDEPEV3'] == 7).sum()
refused = (df['ADDEPEV3'] == 9).sum()

print("--- Excluded Data (ADDEPEV3) ---")
print(f"Missing:    {missing} ({(missing/total)*100:.2f}%)")
print(f"Don't Know: {dont_know} ({(dont_know/total)*100:.2f}%)")
print(f"Refused:    {refused} ({(refused/total)*100:.2f}%)")
print("--------------------------------\n")

# --- Process and calculate prevalence ---
df = df[df['ADDEPEV3'].isin([1, 2])].copy()

df['dep_flag'] = np.where(df['ADDEPEV3'] == 1, 1, 0)
df['weighted_cases'] = df['dep_flag'] * df['_LLCPWT']

# Added include_groups=False to silence the FutureWarning
prevalence = df.groupby('_STATE').apply(
    lambda x: x['weighted_cases'].sum() / x['_LLCPWT'].sum(),
    include_groups=False
).reset_index(name='depression_prevalence')

print(prevalence.head())

--- Excluded Data (ADDEPEV3) ---
Missing:    5 (0.00%)
Don't Know: 2066 (0.45%)
Refused:    593 (0.13%)
--------------------------------

   _STATE  depression_prevalence
0     1.0               0.248037
1     2.0               0.219137
2     4.0               0.191868
3     5.0               0.251524
4     6.0               0.178296
